# Module 1: Neural Network Foundations

This notebook provides hands-on implementations of neural network fundamentals.

**Topics covered:**
- Neuron model and activation functions
- Forward pass computation
- Building a neural network from scratch
- Visualizing decision boundaries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Callable

# Set random seed for reproducibility
np.random.seed(42)

# For nice plots
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1.1 Activation Functions

Activation functions introduce non-linearity into neural networks.

In [ ]:
def sigmoid(x):
    """Sigmoid activation: squashes to (0, 1)"""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def tanh(x):
    """Tanh activation: squashes to (-1, 1)"""
    return np.tanh(x)

def relu(x):
    """ReLU activation: max(0, x)"""
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    """Leaky ReLU: allows small negative values"""
    return np.where(x > 0, x, alpha * x)

def gelu(x):
    """GELU activation: used in Transformers"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

In [ ]:
# Visualize activation functions
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

activations = [
    ('Sigmoid', sigmoid),
    ('Tanh', tanh),
    ('ReLU', relu),
    ('Leaky ReLU', leaky_relu),
    ('GELU', gelu),
]

for idx, (name, func) in enumerate(activations):
    ax = axes.flat[idx]
    ax.plot(x, func(x), 'b-', linewidth=2)
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
    ax.set_title(name, fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.grid(True, alpha=0.3)

# Hide empty subplot
axes.flat[-1].axis('off')

plt.tight_layout()
plt.suptitle('Activation Functions', fontsize=16, y=1.02)
plt.show()

## 1.2 Single Neuron Implementation

A single neuron computes: `y = activation(w @ x + b)`

In [ ]:
class Neuron:
    """A single neuron with configurable activation."""
    
    def __init__(self, n_inputs: int, activation: str = 'relu'):
        """
        Initialize a neuron.
        
        Args:
            n_inputs: Number of input features
            activation: 'relu', 'sigmoid', 'tanh', or 'linear'
        """
        # He initialization for ReLU, Xavier for others
        if activation == 'relu':
            scale = np.sqrt(2.0 / n_inputs)
        else:
            scale = np.sqrt(1.0 / n_inputs)
        
        self.weights = np.random.randn(n_inputs) * scale
        self.bias = 0.0
        
        # Set activation function
        self.activation_name = activation
        self.activation_fn = {
            'relu': relu,
            'sigmoid': sigmoid,
            'tanh': tanh,
            'linear': lambda x: x
        }[activation]
    
    def forward(self, x: np.ndarray) -> float:
        """Compute neuron output."""
        # Linear combination
        z = np.dot(self.weights, x) + self.bias
        # Apply activation
        return self.activation_fn(z)
    
    def __repr__(self):
        return f"Neuron(inputs={len(self.weights)}, activation={self.activation_name})"

In [ ]:
# Test a single neuron
neuron = Neuron(3, activation='relu')
print(f"Neuron: {neuron}")
print(f"Weights: {neuron.weights}")
print(f"Bias: {neuron.bias}")

# Forward pass
x = np.array([1.0, 2.0, 3.0])
output = neuron.forward(x)
print(f"\nInput: {x}")
print(f"Output: {output:.4f}")

## 1.3 Dense Layer Implementation

A dense (fully connected) layer is a collection of neurons.

In [ ]:
class DenseLayer:
    """A fully connected layer."""
    
    def __init__(self, n_inputs: int, n_outputs: int, activation: str = 'relu'):
        """
        Initialize a dense layer.
        
        Args:
            n_inputs: Number of input features
            n_outputs: Number of neurons (output features)
            activation: Activation function name
        """
        # He initialization for ReLU
        if activation == 'relu':
            scale = np.sqrt(2.0 / n_inputs)
        else:
            scale = np.sqrt(1.0 / n_inputs)
        
        # Weight matrix: (n_outputs, n_inputs)
        self.weights = np.random.randn(n_outputs, n_inputs) * scale
        self.biases = np.zeros(n_outputs)
        
        self.activation_name = activation
        self.activation_fn = {
            'relu': relu,
            'sigmoid': sigmoid,
            'tanh': tanh,
            'linear': lambda x: x
        }[activation]
        
        # Cache for backward pass
        self.input = None
        self.z = None  # pre-activation
        self.a = None  # post-activation
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        Forward pass.
        
        Args:
            x: Input array of shape (n_inputs,) or (batch_size, n_inputs)
        
        Returns:
            Output array of shape (n_outputs,) or (batch_size, n_outputs)
        """
        self.input = x
        
        # Handle both single sample and batch
        if x.ndim == 1:
            self.z = np.dot(self.weights, x) + self.biases
        else:
            self.z = np.dot(x, self.weights.T) + self.biases
        
        self.a = self.activation_fn(self.z)
        return self.a
    
    def __repr__(self):
        return f"DenseLayer({self.weights.shape[1]} -> {self.weights.shape[0]}, {self.activation_name})"

In [ ]:
# Test a dense layer
layer = DenseLayer(4, 3, activation='relu')
print(f"Layer: {layer}")
print(f"Weights shape: {layer.weights.shape}")
print(f"Biases shape: {layer.biases.shape}")

# Single sample forward pass
x = np.array([1.0, 2.0, 3.0, 4.0])
output = layer.forward(x)
print(f"\nSingle sample input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Output: {output}")

# Batch forward pass
X_batch = np.random.randn(5, 4)  # 5 samples, 4 features
output_batch = layer.forward(X_batch)
print(f"\nBatch input shape: {X_batch.shape}")
print(f"Batch output shape: {output_batch.shape}")

## 1.4 Neural Network Implementation

A neural network is a stack of layers.

In [ ]:
class NeuralNetwork:
    """A simple feedforward neural network."""
    
    def __init__(self):
        self.layers: List[DenseLayer] = []
    
    def add(self, layer: DenseLayer):
        """Add a layer to the network."""
        self.layers.append(layer)
        return self
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """Forward pass through all layers."""
        output = x
        for layer in self.layers:
            output = layer.forward(output)
        return output
    
    def predict(self, x: np.ndarray) -> np.ndarray:
        """Make predictions (same as forward for now)."""
        return self.forward(x)
    
    def summary(self):
        """Print network architecture."""
        print("Neural Network Summary")
        print("=" * 50)
        total_params = 0
        for i, layer in enumerate(self.layers):
            n_params = layer.weights.size + layer.biases.size
            total_params += n_params
            print(f"Layer {i}: {layer} - {n_params:,} params")
        print("=" * 50)
        print(f"Total parameters: {total_params:,}")

In [ ]:
# Build a simple neural network
network = NeuralNetwork()
network.add(DenseLayer(2, 8, activation='relu'))
network.add(DenseLayer(8, 4, activation='relu'))
network.add(DenseLayer(4, 1, activation='sigmoid'))

network.summary()

# Test forward pass
x = np.array([0.5, 0.5])
output = network.forward(x)
print(f"\nInput: {x}")
print(f"Output: {output}")

## 1.5 XOR Problem

The XOR problem demonstrates why non-linearity is essential.

In [ ]:
# XOR dataset
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

# Visualize XOR problem
plt.figure(figsize=(8, 6))
plt.scatter(X_xor[y_xor == 0, 0], X_xor[y_xor == 0, 1], 
            c='red', s=200, marker='o', label='Class 0')
plt.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], 
            c='blue', s=200, marker='x', label='Class 1')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('XOR Problem - Not Linearly Separable')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.show()

In [ ]:
# Let's manually set weights that solve XOR
# The key insight: we need to create intermediate features

# Layer 1: Create two linear boundaries
# h1 = (x1 + x2 > 0.5)  -> Detects "at least one is 1"
# h2 = (x1 + x2 > 1.5)  -> Detects "both are 1"

xor_network = NeuralNetwork()

# Hidden layer with 2 neurons
hidden = DenseLayer(2, 2, activation='sigmoid')
hidden.weights = np.array([[1, 1], [1, 1]])  # Both check x1 + x2
hidden.biases = np.array([-0.5, -1.5])        # Different thresholds

# Output layer: h1 AND NOT h2
output = DenseLayer(2, 1, activation='sigmoid')
output.weights = np.array([[1, -2]])  # h1 - 2*h2
output.biases = np.array([-0.5])

xor_network.add(hidden)
xor_network.add(output)

# Test on XOR inputs
print("XOR Network Predictions:")
print("Input -> Hidden -> Output -> Rounded")
for x, y in zip(X_xor, y_xor):
    h = hidden.forward(x)
    o = output.forward(h)
    print(f"{x} -> {h.round(2)} -> {o[0]:.3f} -> {int(o[0] > 0.5)} (true: {y})")

In [ ]:
# Visualize the decision boundary
def plot_decision_boundary(network, X, y, title="Decision Boundary"):
    """Plot the decision boundary of a binary classifier."""
    # Create a mesh grid
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predict on mesh
    Z = np.array([network.forward(np.array([x, y_val]))[0] 
                  for x, y_val in zip(xx.ravel(), yy.ravel())])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, levels=50, cmap='RdYlBu', alpha=0.8)
    plt.colorbar(label='Output')
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    
    plt.scatter(X[y == 0, 0], X[y == 0, 1], c='red', s=200, 
                marker='o', edgecolors='black', label='Class 0')
    plt.scatter(X[y == 1, 0], X[y == 1, 1], c='blue', s=200, 
                marker='x', label='Class 1', linewidths=3)
    
    plt.xlabel('x1', fontsize=12)
    plt.ylabel('x2', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend()
    plt.show()

plot_decision_boundary(xor_network, X_xor, y_xor, 
                       "XOR Decision Boundary (Neural Network)")

## 1.6 Parameter Counting

Understanding how many parameters are in a network is crucial.

In [ ]:
def count_parameters(architecture: List[int], include_biases: bool = True) -> int:
    """
    Count parameters in a fully connected network.
    
    Args:
        architecture: List of layer sizes, e.g., [784, 256, 128, 10]
        include_biases: Whether to count bias parameters
    
    Returns:
        Total number of parameters
    """
    total = 0
    print(f"Architecture: {' -> '.join(map(str, architecture))}")
    print("-" * 50)
    
    for i in range(len(architecture) - 1):
        n_in = architecture[i]
        n_out = architecture[i + 1]
        
        weights = n_in * n_out
        biases = n_out if include_biases else 0
        layer_total = weights + biases
        
        print(f"Layer {i+1}: {n_in} x {n_out} = {weights:,} weights", end="")
        if include_biases:
            print(f" + {biases:,} biases = {layer_total:,}")
        else:
            print()
        
        total += layer_total
    
    print("-" * 50)
    print(f"Total: {total:,} parameters")
    return total

# Example: MNIST classifier
print("\nMNIST Classifier:")
count_parameters([784, 256, 128, 10])

print("\n" + "=" * 50 + "\n")

# Example: Larger network
print("Larger Network:")
count_parameters([784, 512, 512, 256, 128, 10])

## Exercises

Try these exercises to solidify your understanding:

In [ ]:
# Exercise 1: Implement the derivative of ReLU
def relu_derivative(x):
    """Compute derivative of ReLU."""
    # YOUR CODE HERE
    pass

# Exercise 2: Implement softmax
def softmax(x):
    """Compute softmax with numerical stability."""
    # YOUR CODE HERE
    # Hint: subtract max for numerical stability
    pass

# Exercise 3: Build a network for binary classification on circles dataset
from sklearn.datasets import make_circles
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.5)

# Build and test your network
# YOUR CODE HERE

## Summary

In this notebook, we:
1. Implemented various activation functions
2. Built a single neuron from scratch
3. Created dense layers and a neural network class
4. Solved the XOR problem
5. Visualized decision boundaries
6. Counted network parameters

**Next:** Module 2 covers training these networks with backpropagation and optimization.